7. Extend the pipeline to handle a 'dirty data' scenario not covered in class (e.g., mixed date formats,
currency symbols in numeric columns) and document your approach.

In [0]:
%python
# Task 7: Extend pipeline to handle dirty data scenarios
# Documented approach for handling:
# 1. Mixed date formats (e.g., "2023-01-15", "01/15/2023", "15-Jan-2023")
# 2. Currency symbols in numeric columns (e.g., "$1,234.56", "€999.99")
# 3. Extra whitespace and inconsistent casing

from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType, DateType
import re

# Example dirty data
dirty_data = [
    ("1", "$1,234.56", "2023-01-15", "  Product A  "),
    ("2", "€2,500.00", "01/15/2023", "product b"),
    ("3", "3456.78", "15-Jan-2023", "PRODUCT C"),
    ("4", "$4,999", "2023/02/20", "Product D"),
    ("5", "5000", "20-02-2023", "  product e  ")
]

df_dirty = spark.createDataFrame(dirty_data, ["id", "amount", "date", "product_name"])

# the dirty data frame 
# df_dirty.show(truncate=False)

cleaned_df = (df_dirty
                   .withColumn("clean_date", coalesce(
                        try_to_date(col("date"), "yyyy-MM-dd"),
                        try_to_date(col("date"), "MM/dd/yyyy"),
                        try_to_date(col("date"), "dd-MMM-yyyy"),
                        try_to_date(col("date"), "yyyy/MM/dd"),
                        try_to_date(col("date"), "dd-MM-yyyy")
                   ))
                   .withColumn("clean_amount" , coalesce(
                        regexp_replace(col("amount") , r"[$₹€£,]", "")
                    ))
                )

cleaned_df.display()



8. Set up a branching strategy (dev/main) for the Cyntexa analytics repo and write a short guide for teammates on the pull-request review workflow before merging into main.

## Branching Strategy – Cyntexa Analytics Repo

We will use two main branches:

- **main** → Production-ready and stable code.
- **dev** → Development and testing branch.

### Workflow

1. Developers create a feature branch from `dev`, for example:
   `feature/customer-cleaning`

2. The developer makes the required changes and tests the notebook/pipeline.

3. The changes are committed and pushed to the remote repository.

4. A Pull Request (PR) is created from the feature branch to `dev`.

5. Another team member reviews the PR and checks:
   - Code quality and readability
   - Data cleaning logic
   - Possible errors or edge cases
   - Whether the changes have been tested

6. If changes are requested, the developer updates the feature branch and pushes the changes.

7. Once the PR is approved and checks pass, it is merged into `dev`.

8. After testing the complete solution in `dev`, a PR is created from `dev` to `main`.

9. The team reviews and approves the PR before merging it into `main`.

### Branch Flow

feature branch → dev → main

This workflow keeps `main` stable and ensures that changes are reviewed and tested before reaching production.

9. (Data Analyst) Using the cleaned dataset, produce a summary report answering 3 business questions
(e.g., top-selling category per region, month-over-month growth, average order value trend) and
note any data-quality caveats a stakeholder should know about.